# Fresh NUDT draw (seed 1010, 150 photos): ours vs the T10 rules, nobody has seen these photos
Dumped on our RTX 3050 after the rules were fixed (frozen T7 judge, cuts 0.9 / 0.98 from T10, our bars 8 / 11 + agreement). Excluded: his 404-808 and 909 draws, our earlier NUDT sets, the T7-seen photos and the 500-pool.
Same stress test as `Ours_vs_T10_Rules`: every inlier count x f for every rule. Paired bootstrap of every rule against ours.

In [ ]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import joblib, json, numpy as np, pandas as pd
pd.set_option("display.width", 220)
from train.checks import replay
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
T7 = ROOT / "tests/Test_Iteration_7/results"
SETS = {"nudt1010": "nudt_fresh1010"}
photos = {s: replay.load_set(T7 / "rich_dump" / f) for s, f in SETS.items()}
from train.checks.decision_signals import build_signal_table
for s, f in SETS.items():
    out = T7 / f"signals_ours_{s}.csv.gz"
    if not out.exists():
        build_signal_table(T7 / "rich_dump" / f, s, 4).to_csv(out, index=False)
raw = pd.concat([pd.read_csv(T7 / f"signals_ours_{s}.csv.gz") for s in SETS], ignore_index=True)
judge = joblib.load(T7 / "judge_t7.joblib")
F = cj.FEATURES_CTX
SCALES = [1.0, 1.25, 1.5, 2.0]
print({s: len(p) for s, p in photos.items()}, len(raw))

In [ ]:
def ours_rule(f):
    rows = []
    for data, ph in photos.items():
        for pid, photo in ph.items():
            inflated = [np.round(np.asarray(q["inliers"], float) * f) for q in photo["queries"]]
            r = replay.decide_current(photo, inliers=inflated)
            rows.append(dict(data=data, photo=pid, stage=r["stage"], pts=int(r["points"]), err_sigma=r["err_sigma"], n_queries=r["n_queries"]))
    return pd.DataFrame(rows)

def merge_or(a, b):
    j = a.set_index(["data", "photo"]).join(b.set_index(["data", "photo"]), rsuffix="_j")
    A, B = j.stage != "skip", j.stage_j != "skip"
    pts = np.where(A, j.pts, np.where(B, j.pts_j, 0))
    st = np.where(A, j.stage, np.where(B, j.stage_j, "skip"))
    return pd.DataFrame(dict(pts=pts, stage=st, n_queries=np.where(A, j.n_queries, np.maximum(j.n_queries, j.n_queries_j))), index=j.index).reset_index()

def judge_rules(t, p):
    out = {}
    out["bars (no agreement)"] = cj.baseline(t, 8, 11, min_choice=3)
    out["J0 judge+override12 (0.9)"] = cj.decide(t, p, 0.9, override=12, wide_tau=0.95, min_choice=3)
    out["J1 judge, no override (0.9)"] = cj.decide(t, p, 0.9, override=None, wide_tau=0.95, min_choice=3)
    out["J2 judge, no override (0.98)"] = cj.decide(t, p, 0.98, override=None, wide_tau=0.98, min_choice=3)
    return out

In [ ]:
import gc
RESULT = {}
for f in SCALES:
    t = sr.scale_signals(raw, f)
    p = sr.predict(judge, t, F)
    r = judge_rules(t, p)
    r["ours (agreement+escalation)"] = ours_rule(f)
    r["ours OR J1"] = merge_or(r["ours (agreement+escalation)"], r["J1 judge, no override (0.9)"])
    RESULT[f] = r
    del t, p; gc.collect()
rows = []
for f, r in RESULT.items():
    for k, v in r.items():
        s = cj.summary(v)
        for data in s.index:
            rows.append(dict(scale=f, rule=k, data=data, score=s.loc[data, "score"], wrong=int(s.loc[data, "wrong"]), answered=s.loc[data, "answered"], queries=s.loc[data, "queries"]))
TAB = pd.DataFrame(rows)
for data in SETS:
    d = TAB[TAB.data == data]
    print(data)
    display(pd.concat({"score": d.pivot_table(index="rule", columns="scale", values="score").round(2), "wrong": d.pivot_table(index="rule", columns="scale", values="wrong").astype(int)}, axis=1))

In [ ]:
# wrong answers per photo at x1 for every rule: which photos are the mistakes?
x1 = RESULT[1.0]
mis = []
for k, v in x1.items():
    w = v[(v.stage != "skip") & (v.pts < 0)] if "pts" in v else v[0:0]
    for _, r in w.iterrows():
        mis.append(dict(rule=k, data=r["data"], photo=r["photo"]))
pd.DataFrame(mis)

In [ ]:
TAB.to_csv(T7 / "ours_vs_t10_fresh1010.csv", index=False)
gains = []
for f, r in RESULT.items():
    for k, v in r.items():
        if k.startswith("ours (") : continue
        g = cj.paired_gain(r["ours (agreement+escalation)"], v, list(SETS))
        gains.append(dict(scale=f, rule=k, **g))
pd.DataFrame(gains).pivot_table(index="rule", columns="scale", values="nudt1010", aggfunc="first")